# Berkeley Parcels × Active Housing Permits

**Goal:** Identify which of Berkeley's 29,024 parcels currently have active housing-related permits.

**Data sources:**
- Parcels: `bhxd-e6up` (City of Berkeley Open Data)
- Zoning Permits: via Socrata API
- Local database: `/Users/johngage/berkeley-data/berkeley.db`

**Strategy:**
1. Load existing parcel data from local SQLite (fast) or re-fetch from API
2. Fetch all active/open housing-related permits from Berkeley Open Data
3. Join on APN to identify which parcels have active permits
4. Summarize and map the results

In [ ]:
# CELL 1: Setup and imports
import pandas as pd
import requests
import sqlite3
import json
import os
from datetime import datetime

print(f"🕐 Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

# Berkeley Open Data Socrata API
BASE_URL = "https://data.cityofberkeley.info/resource"

# Dataset IDs
DATASETS = {
    'parcels': 'bhxd-e6up',
    'business_licenses': 'rwnf-bu3w',
}

# App token (optional but recommended to avoid throttling)
APP_TOKEN = os.environ.get('SOCRATA_APP_TOKEN', '')

# Local database path
DB_PATH = '/Users/johngage/berkeley-data/berkeley.db'

print("✅ Imports loaded")
print(f"📁 Local DB: {DB_PATH}")
print(f"   Exists: {os.path.exists(DB_PATH)}")

In [ ]:
# CELL 2: Load parcels from local database
print("📦 LOADING PARCELS FROM LOCAL DATABASE\n")
print("="*70)

conn = sqlite3.connect(DB_PATH)

# Check what tables exist
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"Tables in database: {tables['name'].tolist()}\n")

# Load parcels
if 'parcels' in tables['name'].values:
    df_parcels = pd.read_sql('SELECT * FROM parcels', conn)
    print(f"✅ Loaded {len(df_parcels):,} parcels from local DB")
    print(f"📋 Columns: {df_parcels.columns.tolist()}")
    print(f"\nSample:")
    display(df_parcels.head(3))
else:
    print("❌ No 'parcels' table found. Will fetch from API in next cell.")
    df_parcels = None

conn.close()

In [ ]:
# CELL 3: (If needed) Fetch all 29,024 parcels from API
# Skip this cell if Cell 2 loaded parcels successfully

if df_parcels is None or len(df_parcels) == 0:
    print("🌐 FETCHING ALL PARCELS FROM BERKELEY OPEN DATA API\n")
    print("="*70)
    
    all_parcels = []
    offset = 0
    limit = 1000
    
    while True:
        url = f"{BASE_URL}/{DATASETS['parcels']}.json"
        params = {
            '$limit': limit,
            '$offset': offset,
        }
        if APP_TOKEN:
            params['$$app_token'] = APP_TOKEN
        
        response = requests.get(url, params=params, timeout=30)
        
        if response.status_code != 200:
            print(f"❌ Error at offset {offset}: {response.status_code}")
            break
        
        batch = response.json()
        if not batch:
            break
        
        all_parcels.extend(batch)
        offset += limit
        print(f"  Fetched {len(all_parcels):,} parcels...")
        
        if len(batch) < limit:
            break
    
    df_parcels = pd.DataFrame(all_parcels)
    print(f"\n✅ Total parcels fetched: {len(df_parcels):,}")
    print(f"📋 Columns: {df_parcels.columns.tolist()}")
    
    # Save to local DB for next time
    conn = sqlite3.connect(DB_PATH)
    df_parcels.to_sql('parcels', conn, if_exists='replace', index=False)
    conn.close()
    print(f"💾 Saved to {DB_PATH}")
else:
    print(f"✅ Already have {len(df_parcels):,} parcels from local DB. Skipping API fetch.")

In [ ]:
# CELL 4: Identify the APN column and standardize
print("🔧 STANDARDIZING APN COLUMN\n")
print("="*70)

# Find the APN column (could be 'APN', 'apn', 'parcel_no', etc.)
apn_candidates = [c for c in df_parcels.columns if 'apn' in c.lower() or 'parcel' in c.lower()]
print(f"Possible APN columns: {apn_candidates}")

# Also check for address columns
addr_candidates = [c for c in df_parcels.columns if 'addr' in c.lower() or 'situs' in c.lower() or 'street' in c.lower()]
print(f"Possible address columns: {addr_candidates}")

# Set the APN column name (adjust if needed)
APN_COL = apn_candidates[0] if apn_candidates else None
ADDR_COL = addr_candidates[0] if addr_candidates else None

if APN_COL:
    print(f"\nUsing APN column: '{APN_COL}'")
    print(f"Sample APNs: {df_parcels[APN_COL].head(5).tolist()}")
    print(f"Unique APNs: {df_parcels[APN_COL].nunique():,}")
    print(f"Null APNs: {df_parcels[APN_COL].isna().sum():,}")
    
    # Standardize: strip whitespace, uppercase
    df_parcels['apn_clean'] = df_parcels[APN_COL].astype(str).str.strip().str.upper()
else:
    print("⚠️ No APN column found! Check column names above.")

if ADDR_COL:
    print(f"\nUsing Address column: '{ADDR_COL}'")
    print(f"Sample: {df_parcels[ADDR_COL].head(3).tolist()}")

In [ ]:
# CELL 5: Fetch active housing permits from Berkeley Open Data
print("🏗️ FETCHING ACTIVE HOUSING PERMITS\n")
print("="*70)

# Berkeley's zoning/building permits may be in several datasets.
# Let's search the catalog first.

catalog_url = "https://data.cityofberkeley.info/api/catalog/v1"
search_terms = ['permit', 'building', 'zoning', 'housing', 'construction']

permit_datasets = []

for term in search_terms:
    try:
        params = {'q': term, 'limit': 10, 'only': 'datasets'}
        response = requests.get(catalog_url, params=params, timeout=10)
        
        if response.status_code == 200:
            results = response.json()
            for result in results.get('results', []):
                resource = result.get('resource', {})
                name = resource.get('name', 'Unknown')
                dataset_id = resource.get('id', 'N/A')
                desc = resource.get('description', '')[:80]
                
                if dataset_id not in [d['id'] for d in permit_datasets]:
                    permit_datasets.append({
                        'name': name,
                        'id': dataset_id,
                        'description': desc,
                        'search_term': term
                    })
    except Exception as e:
        print(f"  ⚠️ Error searching '{term}': {e}")

print(f"Found {len(permit_datasets)} datasets:\n")
for d in permit_datasets:
    print(f"  📁 {d['name']}")
    print(f"     ID: {d['id']}")
    print(f"     {d['description']}")
    print()

In [ ]:
# CELL 6: Fetch permits from discovered datasets
# Update the PERMIT_DATASET_ID below based on what Cell 5 found.
# Common Berkeley permit datasets:
#   - Building permits
#   - Zoning permits
#   - Planning applications

print("🏗️ FETCHING PERMIT RECORDS\n")
print("="*70)

# Try fetching from each permit dataset found above
all_permits = []

for ds in permit_datasets:
    dataset_id = ds['id']
    name = ds['name']
    
    try:
        # First get a small sample to see the columns
        url = f"{BASE_URL}/{dataset_id}.json?$limit=5"
        if APP_TOKEN:
            url += f"&$$app_token={APP_TOKEN}"
        
        response = requests.get(url, timeout=15)
        
        if response.status_code == 200:
            sample = response.json()
            if sample:
                cols = list(sample[0].keys())
                
                # Check if this dataset has address or APN fields
                has_address = any('addr' in c.lower() or 'location' in c.lower() or 'street' in c.lower() for c in cols)
                has_apn = any('apn' in c.lower() or 'parcel' in c.lower() for c in cols)
                has_permit = any('permit' in c.lower() or 'status' in c.lower() or 'type' in c.lower() for c in cols)
                
                print(f"\n📁 {name} ({dataset_id})")
                print(f"   Columns: {cols[:10]}{'...' if len(cols) > 10 else ''}")
                print(f"   Has address: {has_address} | Has APN: {has_apn} | Has permit info: {has_permit}")
                
                if has_permit and (has_address or has_apn):
                    print(f"   ✅ RELEVANT — fetching all records...")
                    
                    # Fetch all records from this dataset
                    records = []
                    offset = 0
                    limit = 1000
                    
                    while True:
                        fetch_url = f"{BASE_URL}/{dataset_id}.json?$limit={limit}&$offset={offset}"
                        if APP_TOKEN:
                            fetch_url += f"&$$app_token={APP_TOKEN}"
                        
                        resp = requests.get(fetch_url, timeout=30)
                        if resp.status_code != 200:
                            break
                        
                        batch = resp.json()
                        if not batch:
                            break
                        
                        records.extend(batch)
                        offset += limit
                        
                        if len(batch) < limit:
                            break
                    
                    df_temp = pd.DataFrame(records)
                    df_temp['_source_dataset'] = name
                    df_temp['_source_id'] = dataset_id
                    all_permits.append(df_temp)
                    print(f"   📊 Fetched {len(df_temp):,} records")
                else:
                    print(f"   ⏭️ Skipping (not relevant)")
            else:
                print(f"\n📁 {name} — empty dataset")
        else:
            print(f"\n📁 {name} — HTTP {response.status_code}")
    
    except Exception as e:
        print(f"\n📁 {name} — Error: {e}")

print(f"\n{'='*70}")
print(f"\n📊 SUMMARY: Found {len(all_permits)} relevant permit datasets")
for df in all_permits:
    src = df['_source_dataset'].iloc[0]
    print(f"   • {src}: {len(df):,} records")

In [ ]:
# CELL 7: Also load housing pipeline projects from local CSV
print("🏠 LOADING HOUSING PIPELINE PROJECTS\n")
print("="*70)

import glob

# Find the most recent housing projects file
patterns = [
    '/Users/johngage/berkeley-data/*housing*project*.csv',
    '/Users/johngage/berkeley-data/*pipeline*.csv',
    '/Users/johngage/berkeley-data/*permits*.csv',
]

found_files = []
for pattern in patterns:
    found_files.extend(glob.glob(pattern))

if found_files:
    print(f"Found {len(found_files)} local data files:\n")
    for f in sorted(found_files, key=os.path.getmtime, reverse=True):
        size = os.path.getsize(f) / 1024
        mtime = datetime.fromtimestamp(os.path.getmtime(f)).strftime('%Y-%m-%d')
        print(f"  📄 {os.path.basename(f)} ({size:.0f} KB, modified {mtime})")
    
    # Load the most recent one
    latest = max(found_files, key=os.path.getmtime)
    df_pipeline = pd.read_csv(latest)
    print(f"\n✅ Loaded: {os.path.basename(latest)}")
    print(f"   {len(df_pipeline)} projects")
    print(f"   Columns: {df_pipeline.columns.tolist()}")
else:
    print("⚠️ No local housing project CSVs found.")
    print("   We'll rely on API permit data from Cell 6.")
    df_pipeline = None

# Also check the SQLite database for housing tables
conn = sqlite3.connect(DB_PATH)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
housing_tables = [t for t in tables['name'].tolist() if any(kw in t.lower() for kw in ['housing', 'permit', 'pipeline', 'project'])]
if housing_tables:
    print(f"\n📊 Housing-related tables in SQLite: {housing_tables}")
    for t in housing_tables:
        count = pd.read_sql(f"SELECT COUNT(*) as n FROM [{t}]", conn).iloc[0]['n']
        print(f"   • {t}: {count:,} rows")
conn.close()

In [ ]:
# CELL 8: Join parcels with permits to find active housing parcels
print("🔗 JOINING PARCELS WITH ACTIVE PERMITS\n")
print("="*70)

# We'll try to join on APN first, then fall back to address matching

# Combine all permit dataframes
if all_permits:
    df_all_permits = pd.concat(all_permits, ignore_index=True)
    print(f"Total permit records: {len(df_all_permits):,}")
    
    # Find APN column in permits
    permit_apn_cols = [c for c in df_all_permits.columns if 'apn' in c.lower() or 'parcel' in c.lower()]
    permit_addr_cols = [c for c in df_all_permits.columns if 'addr' in c.lower() or 'location' in c.lower() or 'street' in c.lower()]
    permit_status_cols = [c for c in df_all_permits.columns if 'status' in c.lower()]
    permit_type_cols = [c for c in df_all_permits.columns if 'type' in c.lower() or 'category' in c.lower() or 'description' in c.lower()]
    
    print(f"\nPermit APN columns: {permit_apn_cols}")
    print(f"Permit address columns: {permit_addr_cols}")
    print(f"Permit status columns: {permit_status_cols}")
    print(f"Permit type columns: {permit_type_cols}")
    
    # Filter for housing-related permits
    # Look for keywords in type/description columns
    housing_keywords = ['residential', 'housing', 'dwelling', 'apartment', 'condo',
                        'ADU', 'accessory', 'multi-family', 'new construction',
                        'addition', 'unit', 'SB 9', 'duplex', 'triplex']
    
    # Check each type/description column for housing keywords
    housing_mask = pd.Series([False] * len(df_all_permits))
    
    for col in permit_type_cols:
        col_text = df_all_permits[col].astype(str).str.lower()
        for kw in housing_keywords:
            housing_mask = housing_mask | col_text.str.contains(kw.lower(), na=False)
    
    df_housing_permits = df_all_permits[housing_mask].copy()
    print(f"\n🏠 Housing-related permits: {len(df_housing_permits):,} out of {len(df_all_permits):,}")
    
    # Filter for active status
    if permit_status_cols:
        status_col = permit_status_cols[0]
        print(f"\n📊 Status distribution ('{status_col}'):")
        print(df_housing_permits[status_col].value_counts().head(15).to_string())
        
        # Active statuses (adjust based on what you see above)
        active_keywords = ['open', 'active', 'issued', 'approved', 'pending',
                          'under review', 'in progress', 'submitted', 'applied']
        
        active_mask = df_housing_permits[status_col].astype(str).str.lower().apply(
            lambda x: any(kw in x for kw in active_keywords)
        )
        
        df_active = df_housing_permits[active_mask].copy()
        print(f"\n✅ ACTIVE housing permits: {len(df_active):,}")
    else:
        df_active = df_housing_permits
        print("\n⚠️ No status column found — using all housing permits")
else:
    print("⚠️ No permit data from API. Using local pipeline data if available.")
    df_active = df_pipeline if df_pipeline is not None else pd.DataFrame()

In [ ]:
# CELL 9: Match active permits to parcels
print("🗺️ MATCHING ACTIVE PERMITS TO PARCELS\n")
print("="*70)

if len(df_active) > 0 and len(df_parcels) > 0:
    
    # Try APN join first
    permit_apn_cols = [c for c in df_active.columns if 'apn' in c.lower() or 'parcel' in c.lower()]
    
    if permit_apn_cols:
        PERMIT_APN = permit_apn_cols[0]
        df_active['apn_clean'] = df_active[PERMIT_APN].astype(str).str.strip().str.upper()
        
        # Join
        df_matched = df_parcels.merge(
            df_active,
            on='apn_clean',
            how='inner',
            suffixes=('_parcel', '_permit')
        )
        
        print(f"✅ Matched {len(df_matched):,} permit-parcel combinations")
        print(f"   Unique parcels with active housing permits: {df_matched['apn_clean'].nunique():,}")
        print(f"   Out of {len(df_parcels):,} total parcels ({df_matched['apn_clean'].nunique()/len(df_parcels)*100:.2f}%)")
    else:
        print("⚠️ No APN column in permits — trying address matching...")
        # Address matching fallback would go here
        df_matched = pd.DataFrame()
    
    # Summary
    if len(df_matched) > 0:
        print(f"\n{'='*70}")
        print(f"\n📊 SUMMARY OF PARCELS WITH ACTIVE HOUSING PERMITS:\n")
        
        # Show sample
        display_cols = [c for c in df_matched.columns if any(kw in c.lower() for kw in 
                        ['apn', 'addr', 'situs', 'status', 'type', 'description', 'unit', 'date'])]
        if display_cols:
            print(df_matched[display_cols[:8]].head(20).to_string())
        else:
            display(df_matched.head(20))
        
        # Save results
        output_path = '/Users/johngage/berkeley-data/parcels_with_active_housing_permits.csv'
        df_matched.to_csv(output_path, index=False)
        print(f"\n💾 Saved to: {output_path}")
        
        # Also save to SQLite
        conn = sqlite3.connect(DB_PATH)
        df_matched.to_sql('parcels_active_housing', conn, if_exists='replace', index=False)
        conn.close()
        print(f"💾 Saved to SQLite table: parcels_active_housing")
else:
    print("⚠️ No active permits or parcels to match.")

In [ ]:
# CELL 10: Quick map of parcels with active housing permits
print("🗺️ MAPPING PARCELS WITH ACTIVE HOUSING PERMITS\n")

try:
    import folium
    from folium.plugins import MarkerCluster
    
    # Berkeley center
    m = folium.Map(location=[37.8716, -122.2727], zoom_start=14, 
                   tiles='CartoDB positron')
    
    # Find lat/lon columns
    lat_cols = [c for c in df_matched.columns if 'lat' in c.lower()]
    lon_cols = [c for c in df_matched.columns if 'lon' in c.lower() or 'lng' in c.lower()]
    
    if lat_cols and lon_cols:
        LAT = lat_cols[0]
        LON = lon_cols[0]
        
        # Filter rows with valid coordinates
        df_map = df_matched.dropna(subset=[LAT, LON]).copy()
        df_map[LAT] = pd.to_numeric(df_map[LAT], errors='coerce')
        df_map[LON] = pd.to_numeric(df_map[LON], errors='coerce')
        df_map = df_map.dropna(subset=[LAT, LON])
        
        print(f"Mapping {len(df_map):,} parcels with coordinates...")
        
        cluster = MarkerCluster()
        
        for _, row in df_map.iterrows():
            addr = row.get(ADDR_COL, row.get('apn_clean', 'Unknown'))
            popup_text = f"<b>{addr}</b><br>APN: {row.get('apn_clean', 'N/A')}"
            
            folium.CircleMarker(
                location=[row[LAT], row[LON]],
                radius=6,
                color='#2d5f3a',
                fill=True,
                fill_color='#4CAF50',
                fill_opacity=0.7,
                popup=folium.Popup(popup_text, max_width=250)
            ).add_to(cluster)
        
        cluster.add_to(m)
        
        # Save map
        map_path = '/Users/johngage/berkeley-data/active_housing_permits_map.html'
        m.save(map_path)
        print(f"\n💾 Map saved to: {map_path}")
        
        display(m)
    else:
        print("⚠️ No coordinate columns found in matched data.")
        print(f"   Available columns: {df_matched.columns.tolist()}")
        print("   You may need to geocode the addresses first.")
        
except ImportError:
    print("⚠️ folium not installed. Run: pip install folium")
    print("   Skipping map — data is still saved to CSV and SQLite.")

In [ ]:
# CELL 11: Final summary
print("\n" + "="*70)
print("📊 FINAL SUMMARY")
print("="*70)
print(f"\n  Total Berkeley parcels:          {len(df_parcels):>8,}")

if 'df_all_permits' in dir() and len(df_all_permits) > 0:
    print(f"  Total permit records fetched:     {len(df_all_permits):>8,}")

if 'df_housing_permits' in dir() and len(df_housing_permits) > 0:
    print(f"  Housing-related permits:          {len(df_housing_permits):>8,}")

if 'df_active' in dir() and len(df_active) > 0:
    print(f"  Active housing permits:           {len(df_active):>8,}")

if 'df_matched' in dir() and len(df_matched) > 0:
    print(f"  Parcels with active permits:      {df_matched['apn_clean'].nunique():>8,}")
    pct = df_matched['apn_clean'].nunique() / len(df_parcels) * 100
    print(f"  Percentage of all parcels:        {pct:>7.2f}%")

print(f"\n  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n{'='*70}")
print("\n✅ Done! Results saved to:")
print(f"   CSV: /Users/johngage/berkeley-data/parcels_with_active_housing_permits.csv")
print(f"   SQLite: {DB_PATH} → table 'parcels_active_housing'")
print(f"   Map: /Users/johngage/berkeley-data/active_housing_permits_map.html")